# 7. Question 3: do detectors built on the embeddings make different errors between groups?

**Detectors.** For every diagnosis (EchoNext: composite SHD and its 11 components; MIMIC-IV: the composite and the six components that passed the ascertainment rule), a detector is an L2-regularised logistic regression on the standardised embeddings, **fitted on all fitting patients regardless of race**, with C chosen by the validation AUROC. **One threshold**, the score that gives 80% sensitivity among the validation patients, is applied to every group. Every patient's most recent ECG is scored once (cross-fitting).

**Errors by group.** At that threshold each group has a sensitivity and a false-positive rate. Differences from White patients are summarised as the **flagging rate** (shifted operating point) and the **balanced accuracy** (separation), directly standardised for age and sex, with 95% CIs from 1000 patient-bootstrap resamples (notebook 00, section 0.7). The resamples of a diagnosis are the same for every model and, in notebook 08, for every eraser.

Solvers (notebook 00, section 0.5): scikit-learn on the CPU for EchoNext (5 folds × 3 models in parallel workers), the GPU solver for MIMIC-IV.

**Paper items:** Fig. 2b,c, Supplementary Tables 5 and 6, Table 2 (questions 3 and 4).

Run time: about 10 minutes for EchoNext (10 CPU workers), 1 minute for MIMIC-IV and 1 minute for the bootstrap.

In [ ]:
%run ./00_settings.ipynb
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

OUT = results("07_disease_detectors")

## 7.1 EchoNext detectors (scikit-learn, CPU)

In [ ]:
D = load_design("echonext")
rows = [fold_rows(D, k) for k in range(N_FOLDS)]
folder = work("echonext", "detectors")


def echonext_worker(fm, k, rows_k, Y):
    """One model, one round: standardise on the fitting rows, then one detector per diagnosis (C by validation AUROC, 80%-sensitivity threshold)."""
    with threadpool_limits(2):
        X = emb_echonext(fm)
        tr, va, ev = rows_k
        sc = StandardScaler().fit(X[tr].astype(np.float64))
        Xtr, Xva, Xev = (sc.transform(X[i].astype(np.float64)) for i in (tr, va, ev))
        del X
        out = {"p_eval": [], "thr": [], "C": [], "val_auc": []}
        for j in range(Y.shape[1]):
            a, Cj, w, b = lr_select_cpu(Xtr, Y[tr, j], Xva, Y[va, j], C_DETECTOR)
            w, b = w.ravel(), float(b[0])
            pv, pe = sigmoid(Xva @ w + b), sigmoid(Xev @ w + b)
            out["p_eval"].append(pe)
            out["thr"].append(float(np.quantile(pv[Y[va, j] == 1], 1 - SENS)))
            out["C"].append(Cj)
            out["val_auc"].append(a)
        return fm, k, {key: np.array(v) for key, v in out.items()}


todo = [fm for fm in FMS if not (folder / f"oof_{fm}.npz").exists()]
if todo:
    t0 = time.time()
    outs = Parallel(n_jobs=CPU_JOBS, backend="loky")(delayed(echonext_worker)(fm, k, rows[k], D["Y_row"]) for fm in todo for k in range(N_FOLDS))
    n_pat = len(D["fold_of"])
    for fm in todo:
        P = np.full((n_pat, len(E_LABELS)), np.nan)
        PRED = np.zeros((n_pat, len(E_LABELS)), np.int8)
        thr = np.zeros((N_FOLDS, len(E_LABELS))); Cs = np.zeros_like(thr); va = np.zeros_like(thr)
        for f, k, res in outs:
            if f == fm:
                pidx = D["pi_of_row"][rows[k][2]]
                P[pidx] = res["p_eval"].T
                PRED[pidx] = (res["p_eval"].T >= res["thr"][None, :]).astype(np.int8)
                thr[k], Cs[k], va[k] = res["thr"], res["C"], res["val_auc"]
        assert np.isfinite(P).all()
        np.savez_compressed(folder / f"oof_{fm}.npz", p=P.astype(np.float32), pred=PRED, thr=thr, C=Cs, val_auc=va, labels=np.array(E_LABELS))
    print(f"EchoNext detectors: {(time.time() - t0) / 60:.1f} min")

## 7.2 MIMIC-IV detectors (GPU solver)

In [ ]:
C = load_mimic_cohort()
Dm = load_design("mimic")
M_LABELS = mimic_labels()
Ym = C[M_LABELS].to_numpy(float)
rows_m = C.row.to_numpy()
folder_m = work("mimic", "detectors")
for fm in FMS:
    f = folder_m / f"oof_{fm}.npz"
    if f.exists():
        continue
    t0 = time.time()
    X = emb_mimic(fm)[rows_m]
    n_pat = len(Dm["fold_of"])
    P = np.full((n_pat, len(M_LABELS)), np.nan); thr = np.zeros((N_FOLDS, len(M_LABELS))); Cs = np.zeros_like(thr); va_auc = np.zeros_like(thr)
    for k in range(N_FOLDS):
        tr, va, ev = fold_rows(Dm, k)
        Xtr, Xva, Xev = (to_dev(X[r]) for r in (tr, va, ev))
        standardize_(Xtr, Xva, Xev)
        for j in range(len(M_LABELS)):
            b = lr_select(Xtr, Ym[tr, j].astype(int), Xva, Ym[va, j].astype(int), C_DETECTOR)
            pv = sigmoid(lr_scores(Xva, b["W"], b["b"]))
            thr[k, j], Cs[k, j], va_auc[k, j] = float(np.quantile(pv[Ym[va, j] == 1], 1 - SENS)), b["C"], b["val"]
            P[Dm["pi_of_row"][ev], j] = sigmoid(lr_scores(Xev, b["W"], b["b"]))
        del Xtr, Xva, Xev
        free_gpu()
    assert np.isfinite(P).all()
    pred = (P >= thr[Dm["fold_of"]]).astype(np.int8)          # threshold of the round in which the patient was evaluated
    np.savez_compressed(f, p=P.astype(np.float32), pred=pred, thr=thr, C=Cs, val_auc=va_auc, labels=np.array(M_LABELS))
    print(f"{fm}: {time.time() - t0:.0f} s")

## 7.3 Discrimination: AUROC over all patients and within the White and Black groups (Supplementary Table 5)

In [ ]:
def discrimination(cohort, labels, Y, g, folder):
    rows_ = []
    for fm in FMS:
        z = np.load(folder / f"oof_{fm}.npz", allow_pickle=True)
        assert list(z["labels"]) == list(labels)
        for j, lab in enumerate(labels):
            y, p, pred = Y[:, j], z["p"][:, j].astype(float), z["pred"][:, j]
            r = {"cohort": cohort, "fm": FMS[fm], "fm_key": fm, "label_key": lab, "label": LABEL_NAME[lab], "n": len(y), "pos": int(y.sum()),
                 "auroc": fast_auc(y, p), "sensitivity": float(pred[y == 1].mean()), "specificity": float(1 - pred[y == 0].mean()),
                 "C": ",".join(f"{c:g}" for c in z["C"][:, j])}
            for gi, gn in enumerate(RACE4):
                m = g == gi
                r[f"auroc_{gn}"] = fast_auc(y[m], p[m])
            rows_.append(r)
    return pd.DataFrame(rows_)


lat_m = C.loc[Dm["lat_idx"]].reset_index(drop=True)
disc = pd.concat([discrimination("EchoNext", E_LABELS, D["Y"], D["g"], folder),
                  discrimination("MIMIC-IV", M_LABELS, lat_m[M_LABELS].to_numpy(float), race_index(lat_m.race6), folder_m)], ignore_index=True)
disc.to_csv(OUT / "discrimination.csv", index=False, encoding="utf-8-sig")
disc[disc.label_key.isin([E_COMP, E_AS, M_COMP, M_AS])][["cohort", "fm", "label", "pos", "auroc", "auroc_white", "auroc_black", "sensitivity", "specificity"]].round(3)

## 7.4 Differences between groups in flagging rate and balanced accuracy (Fig. 2b,c, Supplementary Tables 5 and 6)

One row per cohort, model, diagnosis, comparison (Black, Hispanic or Asian minus White), measure (FNR, FPR, flagging rate, balanced accuracy), standardised for age and sex.

In [ ]:
def predictions(folder, labels):
    out = {}
    for fm in FMS:
        z = np.load(folder / f"oof_{fm}.npz", allow_pickle=True)
        out[fm] = {"none": {lab: z["pred"][:, j] for j, lab in enumerate(labels)}}
    return out


t0 = time.time()
diff = pd.concat([
    difference_table("EchoNext", E_LABELS, D["Y"], D["g"], strata6(D["age"], D["male"]), predictions(folder, E_LABELS)),
    difference_table("MIMIC-IV", M_LABELS, lat_m[M_LABELS].to_numpy(float), race_index(lat_m.race6), strata6(lat_m.age.to_numpy(float), lat_m.male.to_numpy(int)),
                     predictions(folder_m, M_LABELS))], ignore_index=True)
diff.to_csv(OUT / "differences_before_erasure.csv", index=False, encoding="utf-8-sig")
print(f"bootstrap: {(time.time() - t0) / 60:.1f} min")
q = diff[(diff.pair == "black-white") & diff.measure.isin(["FLAG", "BA"]) & diff.label_key.isin([E_COMP, E_AS, M_COMP, M_AS])]
q.assign(value=100 * q.value, lo=100 * q.lo, hi=100 * q.hi)[["cohort", "label", "measure", "fm", "value", "lo", "hi"]].round(1)